# Sensor System — pojedyncze okno analityczne

Wczytanie próbek XYZ, metadanych JSON, zdarzeń jakości i rzadkiej telemetrii przypisanej do tego samego okna.

In [ ]:
from pathlib import Path

from load_analysis_window import load_window

In [ ]:
# Plik z katalogu data obok notebooka:
CSV_PATH = Path('data/line_d-node_1-20:04-19.10.2026.csv')

# Możesz też wskazać dowolny główny plik CSV. Loader sam odnajdzie
# odpowiadające mu pliki .json, .events.csv i .telemetry.csv:
# CSV_PATH = Path('/inna/sciezka/line_h-node_1-11:47-16.10.2026.csv')

window = load_window(CSV_PATH)

In [ ]:
metadata = window['metadata']

summary = {
    'first_sample_time': metadata['first_sample_time_local'],
    'sample_count': len(window['samples']),
    'odr_hz': window['odr_hz'],
    'missing_samples': window['missing_samples'],
    'continuity_breaks': window['continuity_breaks'],
    'telemetry_snapshots': len(window['telemetry']),
    'is_continuous': window['is_continuous'],
}
summary

In [ ]:
if window['is_continuous']:
    print('OK: okno nie zawiera zgłoszonych luk ani przerwań ciągłości.')
else:
    print('UWAGA: okno nie jest ciągłe. Szczegóły znajdują się w events.')

window['events']

In [ ]:
# Podstawowe tablice gotowe do dalszej analizy.
time_s = window['time_s']
x, y, z = window['samples'].T

time_s.shape, x.shape, y.shape, z.shape

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import stft

# Dane
t = np.asarray(window['time_s'], dtype=np.float64)
samples = np.asarray(window['samples'], dtype=np.float64)

x, y, z = samples.T

# ODR deklarowany przez system
odr_config = float(window['odr_hz'])



print(f"Liczba próbek:    {len(t):,}")
print(f"Czas pomiaru:     {t[-1] - t[0]:.3f} s")
print(f"ODR systemowy:    {odr_config:.4f} Hz")
print(f"Ciągłość danych:  {window['is_continuous']}")

## Telemetria okna

`time_from_first_sample_s` jest czasem od pierwszej próbki **tego pliku**, więc korzysta z tej samej osi czasu co `window["time_s"]`. `captured_device_time_us` jest monotonicznym czasem node w obrębie bieżącego `boot_epoch`; samodzielnie nie jest czasem kalendarzowym.

Telemetria jest rzadka (zwykle około jednego snapshotu na okno). Temperatura ADXL355 opisuje temperaturę wewnętrzną sensora w chwili snapshotu. Nie należy automatycznie rozciągać pojedynczego punktu na każdą próbkę ani traktować go jako dokładnej temperatury otoczenia. Jeżeli snapshotów jest kilka, można obserwować trend; interpolację należy stosować świadomie.

In [ ]:
from datetime import datetime, timedelta

telemetry = np.asarray(window['telemetry'], dtype=np.float64)
telemetry_columns = window['telemetry_columns']
telemetry_index = {name: i for i, name in enumerate(telemetry_columns)}

if len(telemetry) == 0:
    print('Brak snapshotu telemetrii w granicach tego okna.')
else:
    telemetry_time_s = telemetry[:, telemetry_index['time_from_first_sample_s']]
    temperature_c = telemetry[:, telemetry_index['adxl355_temperature_c']]
    temperature_raw = telemetry[:, telemetry_index['adxl355_temperature_raw']].astype(np.uint16)
    node_time_us = telemetry[:, telemetry_index['captured_device_time_us']].astype(np.uint64)

    first_sample_time = datetime.fromisoformat(metadata['first_sample_time_local'])
    telemetry_local_time = [
        first_sample_time + timedelta(seconds=float(offset))
        for offset in telemetry_time_s
    ]

    telemetry_rows = [
        {
            'time_from_first_sample_s': float(offset),
            'local_time': local_time.isoformat(),
            'temperature_c': float(temperature),
            'temperature_raw': int(raw),
            'captured_device_time_us': int(device_time),
        }
        for offset, local_time, temperature, raw, device_time in zip(
            telemetry_time_s, telemetry_local_time, temperature_c, temperature_raw, node_time_us
        )
    ]
    display(telemetry_rows)

    telemetry_fig = go.Figure(
        go.Scatter(
            x=telemetry_time_s,
            y=temperature_c,
            mode='lines+markers' if len(telemetry) > 1 else 'markers',
            marker=dict(size=10),
            customdata=temperature_raw,
            hovertemplate=(
                't od pierwszej próbki = %{x:.3f} s<br>'
                'temperatura = %{y:.3f} °C<br>'
                'kod raw = %{customdata}<extra></extra>'
            ),
        )
    )
    telemetry_fig.update_layout(
        title='Temperatura wewnętrzna ADXL355 w czasie okna',
        xaxis_title='Czas od pierwszej próbki okna [s]',
        yaxis_title='Temperatura [°C]',
        template='plotly_dark',
        height=420,
    )
    telemetry_fig.show()

In [ ]:
axes = ['X', 'Y', 'Z']
signals = [x, y, z]
colors = ['#EF553B', '#00CC96', '#636EFA']


def plot_indices(data, max_points=80000):
    """Redukcja liczby punktów z zachowaniem lokalnych min/max."""
    n = len(data)

    if n <= max_points:
        return np.arange(n)

    block = int(np.ceil(2 * n / max_points))
    n_blocks = n // block

    chunks = data[:n_blocks * block].reshape(n_blocks, block)
    offsets = np.arange(n_blocks) * block

    idx_min = offsets + np.argmin(chunks, axis=1)
    idx_max = offsets + np.argmax(chunks, axis=1)

    tail = np.arange(n_blocks * block, n)

    return np.unique(np.concatenate((
        [0], idx_min, idx_max, tail
    )))


fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.045,
    subplot_titles=[f'Oś {a}' for a in axes]
)

for i, (axis, signal, color) in enumerate(
    zip(axes, signals, colors), start=1
):
    idx = plot_indices(signal)

    fig.add_trace(
        go.Scattergl(
            x=t[idx] - t[0],
            y=signal[idx],
            mode='lines',
            name=axis,
            line=dict(color=color, width=1),
            hovertemplate=(
                't = %{x:.3f} s<br>'
                'a = %{y:.6f} m/s²<extra></extra>'
            )
        ),
        row=i,
        col=1
    )

    fig.update_yaxes(
        title_text='a [m/s²]',
        row=i,
        col=1
    )

fig.update_xaxes(
    title_text='Czas [s]',
    row=3,
    col=1
)

fig.update_layout(
    title=(
        f'Przyspieszenie XYZ | '
        f'ODR = {odr_config:.3f} Hz | '
        f'N = {len(t):,}'
    ),
    height=850,
    template='plotly_dark',
    hovermode='x unified',
    showlegend=False,
    dragmode='zoom',
    margin=dict(l=70, r=30, t=100, b=60)
)

fig.show()

In [ ]:
# Parametry STFT
fs = odr_config

WINDOW_SECONDS = 16.0
OVERLAP = 0.75
FMAX = min(100.0, fs / 2)

# Długość FFT: najbliższa potęga dwójki
nperseg = int(2 ** np.round(np.log2(WINDOW_SECONDS * fs)))
nperseg = min(nperseg, len(t))

noverlap = int(nperseg * OVERLAP)

print(f"ODR:        {fs:.4f} Hz")
print(f"NPERSEG:    {nperseg}")
print(f"NOVERLAP:   {noverlap}")
print(f"Delta f:    {fs / nperseg:.4f} Hz")
print(f"Czas okna:  {nperseg / fs:.3f} s")

# Nie wykonujemy STFT przez zgłoszone przerwy ciągłości.
if not window['is_continuous']:
    raise ValueError(
        "Okno zawiera luki lub przerwania ciągłości. "
        "Przed STFT należy wydzielić ciągły fragment próbek."
    )

if len(t) < 16:
    raise ValueError("Za mało próbek do wiarygodnej analizy STFT.")

# Wykresy
fig_stft = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.055,
    subplot_titles=[
        'STFT – X',
        'STFT – Y',
        'STFT – Z'
    ]
)

for i, (axis, signal) in enumerate(zip(axes, signals), start=1):

    f, ts, Zxx = stft(
        signal,
        fs=fs,
        window='hann',
        nperseg=nperseg,
        noverlap=noverlap,
        detrend='constant',
        boundary=None,
        padded=False
    )

    # Moduł STFT w dB względem 1 m/s²
    magnitude_db = 20 * np.log10(
        np.maximum(np.abs(Zxx), 1e-12)
    )

    # Ograniczenie zakresu częstotliwości
    mask = f <= FMAX

    # Ograniczenie liczby kolumn na wykresie.
    # Maksimum w każdym bloku zachowuje krótkie zdarzenia.
    max_columns = 1200
    step = max(1, int(np.ceil(len(ts) / max_columns)))

    if step > 1:
        starts = np.arange(0, len(ts), step)
        magnitude_plot = np.maximum.reduceat(
            magnitude_db[mask, :],
            starts,
            axis=1
        )
        time_plot = ts[starts]
    else:
        magnitude_plot = magnitude_db[mask, :]
        time_plot = ts

    fig_stft.add_trace(
        go.Heatmap(
            x=time_plot,
            y=f[mask],
            z=magnitude_plot,
            coloraxis='coloraxis',
            hovertemplate=(
                't = %{x:.2f} s<br>'
                'f = %{y:.2f} Hz<br>'
                'Poziom = %{z:.1f} dB<extra></extra>'
            )
        ),
        row=i,
        col=1
    )

    fig_stft.update_yaxes(
        title_text='f [Hz]',
        range=[0, FMAX],
        row=i,
        col=1
    )

fig_stft.update_xaxes(
    title_text='Czas [s]',
    row=3,
    col=1
)

fig_stft.update_layout(
    title=(
        f'STFT XYZ | ODR = {fs:.3f} Hz | '
        f'Δf = {fs/nperseg:.3f} Hz'
    ),
    height=1000,
    template='plotly_dark',
    coloraxis=dict(
        colorscale='Turbo',
        colorbar=dict(
            title='dB re 1 m/s²',
            thickness=18
        )
    ),
    margin=dict(l=70, r=100, t=100, b=60)
)

fig_stft.show()